In [1]:
import torch

In [2]:
from transformers import AutoTokenizer, AutoModel, BertModel, BertTokenizer

# bert: bert-base-uncased,
# biobert: dmis-lab/biobert-base-cased-v1.1; use BertTokenizer and BertModel instead of AutoTokenizer and AutoModel
# chemberta v2: DeepChem/ChemBERTa-77M-MTR
model_name = "DeepChem/ChemBERTa-77M-MTR"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, use_safetensors=True)

model.eval()

c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 53/53 [00:00<00:00, 3197.20it/s]
RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MTR
Key                             | Status     | 
--------------------------------+------------+-
regression.out_proj.bias        | UNEXPECTED | 
norm_mean                       | UNEXPECTED | 
regression.dense.weight         | UNEXPECTED | 
regression.dense.bias           | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
regression.out_proj.weight      | UNEXPECTED | 
norm_std                        | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(600, 384, padding_idx=1)
    (token_type_embeddings): Embedding(1, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.144, inplace=False)
    (position_embeddings): Embedding(515, 384, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-2): 3 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.109, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropou

In [3]:
def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp_min(1.0)
    return summed / denom


def get_embedding(smiles: str, device: torch.device | None = None) -> torch.Tensor:
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _model = model.to(device)

    inputs = tokenizer(
        smiles,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = _model(**inputs)

    # Masked mean pooling is usually better than CLS for RoBERTa-style models
    embedding = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
    embedding = torch.nn.functional.normalize(embedding, p=2, dim=-1)

    return embedding.squeeze(0).cpu()

In [4]:
def batch_embeddings(smiles_list, batch_size=32, device: torch.device | None = None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _model = model.to(device)
    _model.eval()

    all_embeddings = []

    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = _model(**inputs)

        batch_emb = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
        batch_emb = torch.nn.functional.normalize(batch_emb, p=2, dim=-1)
        all_embeddings.append(batch_emb.cpu())

    return torch.cat(all_embeddings, dim=0)

In [6]:
import pandas as pd

CSV_PATH = "final_processed_data.csv"
df = pd.read_csv(CSV_PATH)

unique_smiles = sorted(list(set(df["SMILES1"]) | set(df["SMILES2"])))
unique_embeddings = batch_embeddings(unique_smiles)

# Create mapping
smiles_to_emb = dict(zip(unique_smiles, unique_embeddings))
emb_s1 = torch.stack([smiles_to_emb[s] for s in df["SMILES1"]])
emb_s2 = torch.stack([smiles_to_emb[s] for s in df["SMILES2"]])

In [7]:
# --- FUSION (IMPORTANT CHANGE) ---
combined_embeddings = torch.cat([emb_s1, emb_s2], dim=1)

print("X shape:", combined_embeddings.shape)

X shape: torch.Size([63304, 768])


In [8]:
import ast

df["Side Effect Name"] = df["Side Effect Name"].apply(ast.literal_eval)

In [9]:
# --- LABEL EXTRACTION (FIXED) ---

from sklearn.preprocessing import MultiLabelBinarizer
import torch

# Only take the actual label column
mlb = MultiLabelBinarizer()

Y = mlb.fit_transform(df["Side Effect Name"])

Y = torch.tensor(Y, dtype=torch.float32)

print("Y shape:", Y.shape)

Y shape: torch.Size([63304, 963])


In [10]:
# --- SAVE OUTPUTS ---
# 1) Per-pair tensors (if you still want them)
torch.save(combined_embeddings, "X_embeddings.pt")
torch.save(Y, "Y_labels.pt")

# 2) Per-drug embedding dictionary keyed by SMILES (used by other notebooks)
chemberta_dict = dict(zip(unique_smiles, unique_embeddings))
torch.save(chemberta_dict, "chemberta_smiles_embeddings.pt")

print("Saved X_embeddings.pt, Y_labels.pt, chemberta_smiles_embeddings.pt")

Saved X_embeddings.pt, Y_labels.pt, chemberta_smiles_embeddings.pt
